# Exercise: Spurious Correlations in Image Classifiers
### *How a model can be right for the wrong reasons*

---

## Background

Modern neural networks are remarkably good at fitting training data — but this can be a double-edged sword. When the training data contains **spurious correlations** (patterns that are predictive in training but not in general), models can learn to rely on those shortcuts rather than the true causal features.

**The Waterbirds dataset** was specifically constructed to study this problem. It contains images of birds labelled as either *waterbird* (e.g. duck, heron) or *landbird* (e.g. sparrow, woodpecker). Crucially, the dataset was built by compositing bird images onto backgrounds:

| Background | Waterbird images | Landbird images |
|---|---|---|
| Water background | **95%** | 5% |
| Land background | 5% | **95%** |

A model that learns \"water background → waterbird\" would get most training examples correct but would **fail catastrophically on the minority groups** (landbirds on water, waterbirds on land).

**Learning objectives:**
1. Train a ResNet on Waterbirds and observe high average accuracy.
2. Break down accuracy by *group* (bird type × background) to reveal the spurious correlation.
3. Visualise what the model attends to using GradCAM.
4. Understand *why* standard ERM training produces this failure mode.

**Estimated time:** ~45 minutes

---

## Part 0 — Setup

Run the cell below to install and import everything needed. This may take a minute on first run.

In [ ]:
# installations
import subprocess, sys
def pip(*args): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("torch", "torchvision", "datasets", "matplotlib", "numpy", "pandas", "Pillow", "grad-cam")

print("All packages installed")

In [ ]:
# imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

from datasets import load_dataset as hf_load_dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import warnings; warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

---
## Part 1 — Loading the Waterbirds Dataset  *(~5 min)*

We load the dataset directly from [HuggingFace](https://huggingface.co/datasets/grodino/waterbirds), which hosts the standard Waterbirds benchmark.

Each example has:
- `image` — a PIL image of a bird
- `label` — binary label: **0 = landbird**, **1 = waterbird**
- `place` — the **background type**: **0 = land**, **1 = water**

In [ ]:
print("Loading Waterbirds from HuggingFace (grodino/waterbirds)...")
hf_ds = hf_load_dataset("grodino/waterbirds")
print("Done\n")

print("Dataset summary")
for split in ["train", "validation", "test"]:
    print(f"  {split:5s}: {len(hf_ds[split]):,} examples")
print(f"\n  'label' : 0 = landbird,  1 = waterbird")
print(f"  'place' : 0 = land bg,   1 = water bg")

We now wrap the HuggingFace dataset in a PyTorch `Dataset` class. It returns `(image_tensor, label, metadata)` tuples, where `metadata[0]` is the background (`place`) label.

In [ ]:
# PyTorch Dataset wrapper
class WaterbirdsHF(Dataset):
    """
    Wraps a HuggingFace Waterbirds split as a PyTorch Dataset.
    Returns (x, y, metadata) where metadata[:, 0] is the background (place) label.
    Also exposes .y_array and .metadata_array for group analysis.
    """
    def __init__(self, hf_split, transform=None):
        self.data           = hf_split
        self.transform      = transform
        self.y_array        = torch.tensor(hf_split["label"])
        self.metadata_array = torch.tensor([[p] for p in hf_split["place"]])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        img = row["image"].convert("RGB")
        if self.transform:
            img = self.transform(img)
        y    = torch.tensor(row["label"], dtype=torch.long)
        meta = torch.tensor([row["place"]], dtype=torch.long)
        return img, y, meta

print("WaterbirdsHF wrapper defined")

### 1.1 — Explore the group structure

**Question 1:** Before running any model, what would a model that *only* looks at the background predict? What would its accuracy be on the training set? (Think about the percentages in the introduction table.)

> *Write your answer here:*

In [ ]:
# EXERCISE 1.1
# Count the four groups in the TRAINING split.
# Groups are defined by (label, place):
#   Group 0: landbird  on land  background  (label=0, place=0)
#   Group 1: landbird  on water background  (label=0, place=1)
#   Group 2: waterbird on land  background  (label=1, place=0)
#   Group 3: waterbird on water background  (label=1, place=1)
#
# TODO: Create a temporary WaterbirdsHF wrapper (no transform needed).
#       Access .y_array and .metadata_array to get labels and backgrounds.
#       Then compute and print the count and percentage for each group.

tmp_train = WaterbirdsHF(hf_ds["train"])
y_train  = # YOUR CODE HERE
bg_train = # YOUR CODE HERE  # hint: column 0 of metadata_array

group_names = [
    "Landbird on land  (majority)",
    "Landbird on water (minority)",
    "Waterbird on land (minority)",
    "Waterbird on water (majority)",
]

print("Training set group counts:")
for g, (y_val, bg_val) in enumerate([(0,0),(0,1),(1,0),(1,1)]):
    mask  = (y_train == y_val) & (bg_train == bg_val)
    count = mask.sum().item()
    print(f"  Group {g} - {group_names[g]:35s}: {count:5d}  ({100*count/len(y_train):.1f}%)")

In [ ]:
# SOLUTION 1.1
tmp_train = WaterbirdsHF(hf_ds["train"])
y_train  = tmp_train.y_array
bg_train = tmp_train.metadata_array[:, 0]

group_names = [
    "Landbird on land  (majority)",
    "Landbird on water (minority)",
    "Waterbird on land (minority)",
    "Waterbird on water (majority)",
]

print("Training set group counts:")
for g, (y_val, bg_val) in enumerate([(0,0),(0,1),(1,0),(1,1)]):
    mask  = (y_train == y_val) & (bg_train == bg_val)
    count = mask.sum().item()
    print(f"  Group {g} - {group_names[g]:35s}: {count:5d}  ({100*count/len(y_train):.1f}%)")

### 1.2 — Visualise example images

**Your task:** Run the visualisation below. Notice how the background is a strong visual cue — and pay attention to the minority-group examples (waterbird on land, landbird on water).

In [ ]:
# Visualise one example from each group
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("One example from each (label x background) group", fontsize=13, fontweight='bold')

for ax, (y_val, bg_val, title) in zip(axes, [
    (0, 0, "Landbird\non land background"),
    (0, 1, "Landbird\non water background"),
    (1, 0, "Waterbird\non land background"),
    (1, 1, "Waterbird\non water background"),
]):
    idxs = ((y_train == y_val) & (bg_train == bg_val)).nonzero(as_tuple=True)[0]
    sample_idx = idxs[0].item()
    img = hf_ds["train"][sample_idx]["image"].convert("RGB").resize((224, 224))
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    is_minority = (y_val == 1 and bg_val == 0) or (y_val == 0 and bg_val == 1)
    color = '#d62728' if is_minority else '#2ca02c'
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(4)
    ax.set_xticks([]); ax.set_yticks([])

green_patch = mpatches.Patch(color='#2ca02c', label='Majority group (>=95%)')
red_patch   = mpatches.Patch(color='#d62728', label='Minority group (<=5%)')
fig.legend(handles=[green_patch, red_patch], loc='lower center', ncol=2, fontsize=10)
plt.tight_layout()
plt.show()

---
## Part 2 — Training a ResNet-18  *(~15 min)*

We will fine-tune a **ResNet-18** pre-trained on ImageNet using standard **Empirical Risk Minimisation (ERM)** — i.e. minimising average cross-entropy loss over all training examples, treating all examples equally regardless of which group they come from.

In [ ]:
# Data transforms and DataLoaders
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

train_data    = WaterbirdsHF(hf_ds["train"], transform=train_transform)
val_data      = WaterbirdsHF(hf_ds["validation"],   transform=eval_transform)
test_data     = WaterbirdsHF(hf_ds["test"],  transform=eval_transform)
test_data_vis = WaterbirdsHF(hf_ds["test"],  transform=eval_transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_data,   batch_size=64, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False, num_workers=2)

print(f"Train: {len(train_data):,}  |  Val: {len(val_data):,}  |  Test: {len(test_data):,}")

### 2.1 — Build the model

**Your task:** Complete the code below to create a ResNet-18 model. We will replace the final fully-connected layer with a binary classifier.

In [ ]:
# EXERCISE 2.1
# TODO: Create a ResNet-18 with ImageNet pre-trained weights.
#       Replace the final fully-connected layer (model.fc) with
#       a new nn.Linear layer that outputs 2 classes.
#       Move the model to DEVICE.
#
# Hint: resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
#       The original model.fc has an .in_features attribute.

model = # YOUR CODE HERE
model.fc = # YOUR CODE HERE
model = model.to(DEVICE)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# SOLUTION 2.1
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

### 2.2 — Training loop

We train with **standard ERM** for a small number of epochs.

> Tip: If you are on CPU, set `NUM_EPOCHS = 1` to save time. The spurious correlation effect is already visible after one epoch.

In [ ]:
# Training configuration
NUM_EPOCHS   = 3        # reduce to 1 on CPU
LR           = 1e-3
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR,
                      momentum=0.9, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

train_losses, val_accs = [], []

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, n_batches = 0.0, 0

    for i, (x, y, meta) in enumerate(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item(); n_batches += 1

        if (i + 1) % 50 == 0:
            print(f"  Epoch {epoch+1}/{NUM_EPOCHS}  |  batch {i+1}/{len(train_loader)}"
                  f"  |  loss: {running_loss/n_batches:.4f}", end='\r')

    scheduler.step()
    epoch_loss = running_loss / n_batches
    train_losses.append(epoch_loss)

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y, meta in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += y.size(0)
    val_acc = correct / total
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  |  loss: {epoch_loss:.4f}  |  val acc: {val_acc:.3f}")

print("\nTraining complete")

---
## Part 3 — Evaluating Average vs. Worst-Group Accuracy  *(~10 min)*

A model with **spurious correlations** may look excellent by the standard **average accuracy** metric, but will perform poorly on the **minority groups**.

### 3.1 — Per-group accuracy

**Your task:** Complete the evaluation function that computes accuracy separately for each of the four groups.

In [ ]:
# EXERCISE 3.1
# Complete the function below.
# It should iterate over the dataloader, run the model,
# and accumulate correct/total counts for each of the 4 groups.

def evaluate_by_group(model, loader, device):
    """
    Returns a dict mapping group_id -> (correct, total, accuracy).
    Groups:
        0: landbird  on land  (y=0, bg=0)
        1: landbird  on water (y=0, bg=1)
        2: waterbird on land  (y=1, bg=0)
        3: waterbird on water (y=1, bg=1)
    """
    model.eval()
    group_correct = {0: 0, 1: 0, 2: 0, 3: 0}
    group_total   = {0: 0, 1: 0, 2: 0, 3: 0}

    with torch.no_grad():
        for x, y, meta in loader:
            x  = x.to(device)
            y  = y.to(device)
            bg = meta[:, 0].to(device)

            preds        = model(x).argmax(dim=1)
            correct_mask = (preds == y)

            for y_val, bg_val, g in [(0,0,0),(0,1,1),(1,0,2),(1,1,3)]:
                # TODO: Create a boolean mask selecting examples in group g
                group_mask = # YOUR CODE HERE
                # TODO: Accumulate correct predictions and totals for this group
                group_correct[g] += # YOUR CODE HERE
                group_total[g]   += # YOUR CODE HERE

    results = {}
    for g in range(4):
        acc = group_correct[g] / group_total[g] if group_total[g] > 0 else 0.0
        results[g] = (group_correct[g], group_total[g], acc)
    return results

print("Function defined. Run the next cell to evaluate.")

In [ ]:
# SOLUTION 3.1
def evaluate_by_group(model, loader, device):
    model.eval()
    group_correct = {0: 0, 1: 0, 2: 0, 3: 0}
    group_total   = {0: 0, 1: 0, 2: 0, 3: 0}

    with torch.no_grad():
        for x, y, meta in loader:
            x  = x.to(device)
            y  = y.to(device)
            bg = meta[:, 0].to(device)

            preds        = model(x).argmax(dim=1)
            correct_mask = (preds == y)

            for y_val, bg_val, g in [(0,0,0),(0,1,1),(1,0,2),(1,1,3)]:
                group_mask = (y == y_val) & (bg == bg_val)
                group_correct[g] += (correct_mask & group_mask).sum().item()
                group_total[g]   += group_mask.sum().item()

    results = {}
    for g in range(4):
        acc = group_correct[g] / group_total[g] if group_total[g] > 0 else 0.0
        results[g] = (group_correct[g], group_total[g], acc)
    return results

print("Function defined.")

In [ ]:
# Evaluate on the test set
print("Evaluating on test set... (this may take a minute)")
test_results = evaluate_by_group(model, test_loader, DEVICE)

all_correct = sum(v[0] for v in test_results.values())
all_total   = sum(v[1] for v in test_results.values())

print("\n" + "="*65)
print(f"{'GROUP':40s}  {'N':>5}  {'ACCURACY':>10}")
print("-"*65)

worst_acc = min(v[2] for v in test_results.values())
for g, name in enumerate(group_names):
    correct, total, acc = test_results[g]
    flag = "  <- worst" if acc == worst_acc else ""
    print(f"{name:40s}  {total:>5}  {acc:>10.1%}{flag}")

print("-"*65)
print(f"{'Average accuracy':40s}  {all_total:>5}  {all_correct/all_total:>10.1%}")
print(f"{'Worst-group accuracy':40s}  {'':>5}  {worst_acc:>10.1%}")
print("="*65)

**Question 2:**
1. What is the gap between average accuracy and worst-group accuracy?
2. Which two groups suffer the most? What do they have in common?
3. Why does ERM lead to this outcome? Think about the loss function and the training distribution.

> *Write your answers here:*

### 3.2 — Visualise the disparity

In [ ]:
# EXERCISE 3.2
# Plot a bar chart showing per-group accuracy for the test set.
# Colour majority groups green and minority groups red.
# Add a horizontal dashed line for average accuracy.
# Add a horizontal dotted line for worst-group accuracy.
#
# TODO: Fill in the two axhline calls.

accs   = [test_results[g][2] for g in range(4)]
sizes  = [test_results[g][1] for g in range(4)]
colors = ['#2ca02c', '#d62728', '#d62728', '#2ca02c']
short_names = [
    "Landbird\non land",
    "Landbird\non water",
    "Waterbird\non land",
    "Waterbird\non water",
]
avg_acc   = all_correct / all_total
worst_acc = min(accs)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(short_names, accs, color=colors, edgecolor='black', linewidth=0.8)

# TODO: add average accuracy line (blue dashed) and worst-group line (orange dotted)
ax.axhline( # YOUR CODE HERE - average accuracy, color='steelblue', linestyle='--'
ax.axhline( # YOUR CODE HERE - worst-group accuracy, color='darkorange', linestyle=':'

for bar, acc, n in zip(bars, accs, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.01,
            f"{acc:.1%}\n(n={n})", ha='center', va='bottom', fontsize=9)

ax.set_ylim(0, 1.18)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("ResNet-18 (ERM): Per-group accuracy on Waterbirds test set",
             fontsize=12, fontweight='bold')

green_p = mpatches.Patch(color='#2ca02c', label='Majority group')
red_p   = mpatches.Patch(color='#d62728', label='Minority group')
ax.legend(handles=[green_p, red_p], loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# SOLUTION 3.2
accs   = [test_results[g][2] for g in range(4)]
sizes  = [test_results[g][1] for g in range(4)]
colors = ['#2ca02c', '#d62728', '#d62728', '#2ca02c']
short_names = [
    "Landbird\non land",
    "Landbird\non water",
    "Waterbird\non land",
    "Waterbird\non water",
]
avg_acc   = all_correct / all_total
worst_acc = min(accs)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(short_names, accs, color=colors, edgecolor='black', linewidth=0.8)

ax.axhline(avg_acc,   color='steelblue',  linestyle='--',  linewidth=2,
           label=f"Average acc: {avg_acc:.1%}")
ax.axhline(worst_acc, color='darkorange', linestyle=':',   linewidth=2.5,
           label=f"Worst-group acc: {worst_acc:.1%}")

for bar, acc, n in zip(bars, accs, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.01,
            f"{acc:.1%}\n(n={n})", ha='center', va='bottom', fontsize=9)

ax.set_ylim(0, 1.18)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("ResNet-18 (ERM): Per-group accuracy on Waterbirds test set",
             fontsize=12, fontweight='bold')

green_p = mpatches.Patch(color='#2ca02c', label='Majority group')
red_p   = mpatches.Patch(color='#d62728', label='Minority group')
lines   = ax.get_lines()
ax.legend(handles=[green_p, red_p, lines[0], lines[1]],
          loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

---
## Part 4 — Where Is the Model Looking? GradCAM  *(~10 min)*

**GradCAM** (Gradient-weighted Class Activation Mapping) produces a heatmap highlighting which spatial regions of an image most influenced the model's prediction. We will compare GradCAM maps for majority-group vs. minority-group examples to see whether the model focuses on the **bird** or the **background**.

In [ ]:
# GradCAM setup
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

target_layers = [model.layer4[-1]]   # last conv block of ResNet-18
cam = GradCAM(model=model, target_layers=target_layers)

inv_normalize = transforms.Normalize(
    mean=[-m/s for m, s in zip([0.485,0.456,0.406],[0.229,0.224,0.225])],
    std =[1/s   for s in [0.229, 0.224, 0.225]]
)

def get_gradcam(dataset_split, group_y, group_bg, n=3):
    """Return n GradCAM visualisations for the given (label, background) group."""
    idxs = ((dataset_split.y_array == group_y) &
            (dataset_split.metadata_array[:, 0] == group_bg)).nonzero(as_tuple=True)[0][:n]
    results = []
    for idx in idxs:
        x_tensor, y_label, _ = dataset_split[idx.item()]
        x_batch = x_tensor.unsqueeze(0).to(DEVICE)
        grayscale_cam = cam(input_tensor=x_batch,
                            targets=[ClassifierOutputTarget(y_label.item())])[0]
        rgb = inv_normalize(x_tensor).permute(1, 2, 0).clamp(0, 1).numpy()
        vis = show_cam_on_image(rgb, grayscale_cam, use_rgb=True)
        results.append((rgb, vis, y_label.item()))
    return results

print("GradCAM helper ready.")

In [ ]:
# EXERCISE 4
# Use get_gradcam() to visualise GradCAM maps for:
#   (a) MAJORITY: waterbirds on water (y=1, bg=1) - the model should do well
#   (b) MINORITY: waterbirds on land  (y=1, bg=0) - the model may struggle
#
# For each group, show 3 examples side by side.
# Each example should show the original image and its GradCAM heatmap.
#
# TODO: Call get_gradcam for groups (a) and (b), then create the plot.

# YOUR CODE HERE
#
# Starter hints:
# majority_results = get_gradcam(test_data_vis, group_y=1, group_bg=1)
# minority_results = get_gradcam(test_data_vis, group_y=1, group_bg=0)
#
# fig, axes = plt.subplots(4, 3, ...)  -- 4 rows x 3 columns
# (row 0: majority originals, row 1: majority GradCAM,
#  row 2: minority originals, row 3: minority GradCAM)

In [ ]:
# SOLUTION 4
majority_results = get_gradcam(test_data_vis, group_y=1, group_bg=1)  # waterbird on water
minority_results = get_gradcam(test_data_vis, group_y=1, group_bg=0)  # waterbird on land

label_names = {0: 'landbird', 1: 'waterbird'}
N = 3

fig, axes = plt.subplots(4, N, figsize=(4*N, 13))
fig.suptitle(
    "GradCAM: What does the model attend to?\n"
    "Rows 1-2: waterbird on water (majority) | Rows 3-4: waterbird on land (minority)",
    fontsize=12, fontweight='bold'
)

for col, (rgb, cam_vis, true_label) in enumerate(majority_results):
    axes[0, col].imshow(rgb)
    axes[0, col].set_title(f"Original | true={label_names[true_label]}", fontsize=8)
    axes[0, col].axis('off')
    axes[1, col].imshow(cam_vis)
    axes[1, col].set_title("GradCAM", fontsize=8)
    axes[1, col].axis('off')

for col, (rgb, cam_vis, true_label) in enumerate(minority_results):
    axes[2, col].imshow(rgb)
    axes[2, col].set_title(f"Original | true={label_names[true_label]}", fontsize=8)
    axes[2, col].axis('off')
    axes[3, col].imshow(cam_vis)
    axes[3, col].set_title("GradCAM", fontsize=8)
    axes[3, col].axis('off')

fig.text(0.01, 0.72, "MAJORITY\n(waterbird\non water)", va='center', ha='left',
         fontsize=10, fontweight='bold', color='#2ca02c', rotation=90)
fig.text(0.01, 0.27, "MINORITY\n(waterbird\non land)", va='center', ha='left',
         fontsize=10, fontweight='bold', color='#d62728', rotation=90)

plt.tight_layout(rect=[0.04, 0, 1, 0.95])
plt.show()

**Question 3:**
1. In the majority-group images, where does GradCAM show the model attending?
2. In the minority-group images, does the attention shift? What regions are highlighted?
3. What does this tell you about what the model has *actually* learned?

> *Write your answers here:*

---
## Part 5 — Reflection and Discussion  *(~5 min)*

### 5.1 — The background-only rule

**Your task:** Compute the accuracy of a trivial rule that ignores the bird entirely and predicts label = background.

In [ ]:
# EXERCISE 5.1
# Compute the accuracy of the background-only rule:
#   predict waterbird (1) if background is water (1), else predict landbird (0).
#
# Compare its per-group accuracy with the model's.
# This quantifies how much of the model's performance comes from the shortcut.
#
# TODO: Get y and bg arrays for the test set, then define bg_rule_preds.

y_test  = test_data.y_array
bg_test = test_data.metadata_array[:, 0]

bg_rule_preds = # YOUR CODE HERE  # predict y = bg (waterbird if water background)

print("Background-only rule - per-group accuracy:")
for g, (y_val, bg_val) in enumerate([(0,0),(0,1),(1,0),(1,1)]):
    mask = (y_test == y_val) & (bg_test == bg_val)
    acc  = (bg_rule_preds[mask] == y_test[mask]).float().mean().item()
    print(f"  {group_names[g]:40s}: {acc:.1%}")

overall = (bg_rule_preds == y_test).float().mean().item()
print(f"\n  {'Overall accuracy of background rule':40s}: {overall:.1%}")

In [ ]:
# SOLUTION 5.1
y_test  = test_data.y_array
bg_test = test_data.metadata_array[:, 0]

bg_rule_preds = bg_test.long()  # predict waterbird (1) if water background (1)

print("Background-only rule - per-group accuracy:")
for g, (y_val, bg_val) in enumerate([(0,0),(0,1),(1,0),(1,1)]):
    mask = (y_test == y_val) & (bg_test == bg_val)
    acc  = (bg_rule_preds[mask] == y_test[mask]).float().mean().item()
    print(f"  {group_names[g]:40s}: {acc:.1%}")

overall = (bg_rule_preds == y_test).float().mean().item()
print(f"\n  {'Overall accuracy of background rule':40s}: {overall:.1%}")

### 5.2 — Final discussion questions

**Question 4:**
The background-only rule achieves surprisingly high *average* accuracy. What does this tell us about why ERM learns the spurious correlation?

> *Your answer:*

---

**Question 5:**
List **two or three strategies** that could mitigate spurious correlations. For each, briefly explain the core idea.

*(Hint: think about reweighting, data augmentation, regularisation, or subgroup-aware objectives.)*

> *Your answer:*

---

**Question 6 (bonus):**
Why is **worst-group accuracy** a better metric than average accuracy for safety-critical applications? Give a real-world example where reporting only average accuracy would be misleading.

> *Your answer:*

---
## Summary

| Metric | Value |
|---|---|
| Average test accuracy | *fill in* |
| Worst-group test accuracy | *fill in* |
| Background-only rule accuracy | *fill in* |
| Gap (avg - worst) | *fill in* |

### Key takeaways

1. **ERM can produce highly accurate models that rely on shortcuts.** High average accuracy does not guarantee robustness.
2. **Spurious correlations are invisible to aggregate metrics.** Disaggregating by group reveals the problem.
3. **Worst-group accuracy is the right objective** when all groups must be served reliably.
4. **GradCAM can expose what a model has actually learned**, helping diagnose shortcut behaviour.

### Further reading

- Sagawa et al. (2020). *Distributionally Robust Neural Networks for Group Shifts.* (Introduces GroupDRO — a principled fix.)
- Geirhos et al. (2020). *Shortcut learning in deep neural networks.* Nature Machine Intelligence.
- Waterbirds on HuggingFace: https://huggingface.co/datasets/grodino/waterbirds

---
*End of exercise.*